# 🚚 Traveling Salesperson Problem (TSP)
## Green Logistics & Sustainable Supply Chain Management

---

| Field | Details |
|---|---|
| **Course** | CSC 301: Algorithm Analysis and Design |
| **Academic Year** | 2025/2026 — 2nd Term |
| **Group** | G1 — Section 6FA1 |
| **Members** | Maisoon Mohammed Alabdullah *(Leader)*, Wajd Khalid Alkhaldi, Jolie Jamal Salama, Nabaa Nabeeh Alaswad, Manar Saleh Alghamdi |
| **Dataset** | Kaggle — Traveling Salesman Problem (mexwell) |
| **Dataset Link** | https://www.kaggle.com/datasets/mexwell/traveling-salesman-problem |
| **Algorithms** | Nearest Neighbor (Greedy) · Held-Karp (Dynamic Programming) |

---


## 1. Introduction

The **Traveling Salesperson Problem (TSP)** is one of the most well-known combinatorial optimization problems in computer science. Given a list of cities and the distances between each pair, the goal is to find the **shortest possible route** that visits each city exactly once and returns to the starting depot.

In the context of **Green Logistics**, solving TSP efficiently is critical because:
- Shorter routes → **less fuel consumption**
- Less fuel → **lower operational costs**
- Lower emissions → **reduced carbon footprint**

### Algorithms Compared

| Algorithm | Paradigm | Time Complexity | Space Complexity | Optimality |
|---|---|---|---|---|
| **Nearest Neighbor** | Greedy | O(n²) | O(n) | Approximate |
| **Held-Karp** | Dynamic Programming | O(2ⁿ · n²) | O(2ⁿ · n) | Guaranteed Optimal |

### Dataset Files Used

| File | Cities | Used For |
|---|---|---|
| `tiny.csv` | 10 cities | Held-Karp — Best Case |
| `small.csv` | 30 cities | Greedy — Best Case · Held-Karp — Average (15 cities) |
| `medium.csv` | 100 cities | Greedy — Average Case · Held-Karp — Worst (20 cities) |
| `large.csv` | 1000 cities | Greedy — Worst Case |


## 2. Libraries & Dependencies

In [ ]:
# ─────────────────────────────────────────────
# Install required libraries (run once if needed)
# ─────────────────────────────────────────────
# !pip install numpy matplotlib pandas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time
import math

print("✅ All libraries imported successfully.")
print(f"   NumPy   version : {np.__version__}")
print(f"   Pandas  version : {pd.__version__}")
print(f"   Matplotlib ver  : {plt.matplotlib.__version__}")


## 3. Dataset Loading

### Source
**Dataset:** Traveling Salesman Problem — Kaggle (mexwell)  
**Link:** https://www.kaggle.com/datasets/mexwell/traveling-salesman-problem

### Structure
Each CSV file contains city coordinates with:
- **No header row**
- **Column 1 (x):** X coordinate (float, scientific notation)
- **Column 2 (y):** Y coordinate (float, scientific notation)

### Instructions
> ⚠️ Place the 4 CSV files (`tiny.csv`, `small.csv`, `medium.csv`, `large.csv`) in the **same folder** as this notebook before running.


In [ ]:
# ─────────────────────────────────────────────
# LOAD ALL DATASET FILES
# ─────────────────────────────────────────────

def load_cities(filepath):
    """
    Loads city coordinates from a CSV file with no headers.

    Parameters:
        filepath (str): Path to the CSV file

    Returns:
        list of (x, y) tuples representing city coordinates
    """
    df = pd.read_csv(filepath, header=None, usecols=[0, 1])
    df.columns = ['x', 'y']
    cities = list(zip(df['x'].astype(float), df['y'].astype(float)))
    return cities

# Load all 4 datasets
tiny_cities   = load_cities('tiny.csv')      # 10   cities
small_cities  = load_cities('small.csv')     # 30   cities
medium_cities = load_cities('medium.csv')    # 100  cities
large_cities  = load_cities('large.csv')     # 1000 cities

# Summary
print("=" * 50)
print("  DATASET LOADING SUMMARY")
print("=" * 50)
print(f"  {'File':<15} : {'Cities':>6}")
print(f"  {'─'*15}───{'─'*6}")
print(f"  {'tiny.csv':<15} : {len(tiny_cities):>6} ✅")
print(f"  {'small.csv':<15} : {len(small_cities):>6} ✅")
print(f"  {'medium.csv':<15} : {len(medium_cities):>6} ✅")
print(f"  {'large.csv':<15} : {len(large_cities):>6} ✅")
print("=" * 50)


### Dataset Visualization — All 4 City Maps

In [ ]:
# ─────────────────────────────────────────────
# VISUALIZE: All 4 datasets side by side
# ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.patch.set_facecolor('#1a1a2e')

titles       = ['Tiny (10 cities)', 'Small (30 cities)',
                'Medium (100 cities)', 'Large (1000 cities)']
colors       = ['#00d4ff', '#2ecc71', '#f39c12', '#e74c3c']
dataset_list = [tiny_cities, small_cities, medium_cities, large_cities]

for ax, data, title, color in zip(axes, dataset_list, titles, colors):
    coords = np.array(data)
    ax.set_facecolor('#16213e')
    ax.scatter(coords[:, 0], coords[:, 1], color=color, s=30,
               edgecolors='white', linewidths=0.5, alpha=0.85)
    ax.set_title(title, color='white', fontsize=11, fontweight='bold', pad=10)
    ax.tick_params(colors='gray')
    ax.spines[:].set_color('#0f3460')

plt.suptitle("Kaggle TSP Dataset — All City Maps",
             color='white', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("\n✅ All 4 datasets visualized.")


## 4. Distance Matrix

We compute the **Euclidean distance** between every pair of cities:

$$d(i,j) = \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2}$$


In [ ]:
# ─────────────────────────────────────────────
# FUNCTION: Compute Euclidean Distance Matrix
# ─────────────────────────────────────────────

def compute_distance_matrix(coords):
    """
    Computes the full pairwise Euclidean distance matrix.

    Parameters:
        coords (list of tuples): City coordinates [(x0,y0), (x1,y1), ...]

    Returns:
        np.ndarray: Symmetric distance matrix of shape (n, n)
    """
    n    = len(coords)
    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            dx = coords[i][0] - coords[j][0]
            dy = coords[i][1] - coords[j][1]
            dist[i][j] = math.sqrt(dx*dx + dy*dy)
    return dist

# Pre-compute distance matrices for all datasets
print("Computing distance matrices ...")
tiny_dist   = compute_distance_matrix(tiny_cities)
small_dist  = compute_distance_matrix(small_cities)
medium_dist = compute_distance_matrix(medium_cities)
large_dist  = compute_distance_matrix(large_cities)
print("✅ All distance matrices computed.")

# Display a 5x5 sample of the tiny distance matrix
print(f"\nDistance Matrix Sample — Tiny Dataset (5x5):")
print(f"\n{'':>8}", end="")
for j in range(5):
    print(f"  City {j}", end="")
print()
print("─" * 50)
for i in range(5):
    print(f"City {i}  |", end="")
    for j in range(5):
        print(f"  {tiny_dist[i][j]:>6.4f}", end="")
    print()


---
## 5. Algorithm 1 — Nearest Neighbor (Greedy)

### How It Works
The **Nearest Neighbor** algorithm is a greedy heuristic for TSP:

1. Start at a chosen depot city
2. At each step, move to the **nearest unvisited city**
3. Repeat until all cities are visited
4. Return to the starting depot

### Complexity
| Metric | Value |
|---|---|
| **Time Complexity** | O(n²) |
| **Space Complexity** | O(n) |
| **Optimality** | Approximate — not guaranteed optimal |
| **Best For** | Large-scale, real-time routing |

### Example Sets
| Case | Dataset | Cities | Reason |
|---|---|---|---|
| **Best** | small.csv | 30 | Starting city yielding shortest route |
| **Average** | medium.csv | 100 | Starting city yielding mid-range route |
| **Worst** | large.csv | 1000 | Starting city yielding longest route |


In [ ]:
# ─────────────────────────────────────────────
# ALGORITHM 1: NEAREST NEIGHBOR (GREEDY)
# ─────────────────────────────────────────────

def nearest_neighbor(dist_matrix, start_city):
    """
    Nearest Neighbor greedy algorithm for TSP.

    Parameters:
        dist_matrix (np.ndarray): Pairwise distance matrix
        start_city  (int)       : Index of the starting/depot city

    Returns:
        tour           (list) : Ordered city indices (returns to start)
        total_distance (float): Total tour distance
    """
    n         = len(dist_matrix)
    unvisited = list(range(n))
    unvisited.remove(start_city)

    tour           = [start_city]
    current        = start_city
    total_distance = 0

    # Greedily pick the nearest unvisited city at each step
    while unvisited:
        nearest = min(unvisited, key=lambda c: dist_matrix[current][c])
        total_distance += dist_matrix[current][nearest]
        tour.append(nearest)
        unvisited.remove(nearest)
        current = nearest

    # Return to starting depot to complete the tour
    total_distance += dist_matrix[current][start_city]
    tour.append(start_city)

    return tour, total_distance


def find_greedy_cases(dist_matrix):
    """
    Runs Nearest Neighbor from every starting city and
    identifies the best, average, and worst results.
    """
    results = []
    for start in range(len(dist_matrix)):
        tour, dist = nearest_neighbor(dist_matrix, start)
        results.append((dist, start, tour))
    results.sort(key=lambda x: x[0])
    best    = results[0]
    worst   = results[-1]
    average = results[len(results) // 2]
    return best, average, worst

print("✅ Nearest Neighbor algorithm defined.")


### Algorithm 1 — Run Best / Average / Worst Cases

In [ ]:
# ─────────────────────────────────────────────
# RUN NEAREST NEIGHBOR — 3 Example Sets
# ─────────────────────────────────────────────

greedy_configs = [
    (small_cities,  small_dist,  "best",    "small.csv  — 30 Cities"),
    (medium_cities, medium_dist, "average", "medium.csv — 100 Cities"),
    (large_cities,  large_dist,  "worst",   "large.csv  — 1000 Cities"),
]

greedy_results = []

print("=" * 68)
print("  NEAREST NEIGHBOR (GREEDY) — Results Summary")
print("=" * 68)
print(f"  {'Case':<10} | {'Dataset':<25} | {'Cities':>6} | {'Distance':>12} | {'Time':>10}")
print(f"  {'─'*10}─{'─'*25}─{'─'*8}─{'─'*14}─{'─'*10}")

for coords, dist_matrix, case_label, dataset_name in greedy_configs:
    best, average, worst = find_greedy_cases(dist_matrix)
    case_map             = {"best": best, "average": average, "worst": worst}
    dist_val, start, _   = case_map[case_label]

    t0             = time.time()
    tour, dist_val = nearest_neighbor(dist_matrix, start)
    exec_time      = time.time() - t0

    greedy_results.append({
        "coords"   : coords,
        "tour"     : tour,
        "distance" : dist_val,
        "case"     : case_label,
        "dataset"  : dataset_name,
        "start"    : start,
        "exec_time": exec_time,
        "n_cities" : len(coords)
    })

    print(f"  {case_label.capitalize():<10} | {dataset_name:<25} | {len(coords):>6} | {dist_val:>12.6f} | {exec_time*1000:>8.3f} ms")

print("=" * 68)


### Visualization Function

In [ ]:
# ─────────────────────────────────────────────
# VISUALIZATION HELPER FUNCTION
# ─────────────────────────────────────────────

CASE_COLORS = {
    "best"   : "#2ecc71",
    "average": "#f39c12",
    "worst"  : "#e74c3c"
}

def plot_tour(coords, tour, title, total_dist, case_label, algo_name, exec_time=None):
    """
    Plots a TSP tour on a styled dark background map.
    """
    coords = np.array(coords)
    color  = CASE_COLORS[case_label]

    fig, ax = plt.subplots(figsize=(11, 8))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#16213e')

    # Draw tour edges
    for i in range(len(tour) - 1):
        x_vals = [coords[tour[i]][0], coords[tour[i+1]][0]]
        y_vals = [coords[tour[i]][1], coords[tour[i+1]][1]]
        ax.plot(x_vals, y_vals, color=color, alpha=0.55, linewidth=1.2, zorder=1)

    # Draw city nodes
    node_size = max(5, 60 - len(coords) // 20)
    ax.scatter(coords[:, 0], coords[:, 1], color='white', s=node_size,
               zorder=3, edgecolors=color, linewidths=0.8)

    # Highlight depot city with a star
    start = tour[0]
    ax.scatter(coords[start][0], coords[start][1], color=color,
               s=220, zorder=4, marker='*', edgecolors='white', linewidths=1.5)

    # Label cities for small sets
    if len(coords) <= 30:
        for i, (x, y) in enumerate(coords):
            ax.annotate(str(i), (x, y), textcoords="offset points",
                        xytext=(5, 5), fontsize=7.5, color='#cccccc')

    # Info box
    info_lines = [
        f"Total Distance : {total_dist:.6f}",
        f"Cities         : {len(coords)}",
        f"Start City     : {start}",
    ]
    if exec_time is not None:
        info_lines.append(f"Exec Time      : {exec_time*1000:.3f} ms")

    ax.text(0.02, 0.98, "\n".join(info_lines),
            transform=ax.transAxes, fontsize=9.5,
            verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.6', facecolor='#0f3460', alpha=0.85),
            color='white', fontfamily='monospace')

    case_display = case_label.capitalize() + " Case"
    ax.set_title(f"{algo_name}\n{case_display}  —  {title}",
                 fontsize=13, color='white', fontweight='bold', pad=14)
    ax.set_xlabel("X Coordinate", color='gray', fontsize=10)
    ax.set_ylabel("Y Coordinate", color='gray', fontsize=10)
    ax.tick_params(colors='gray')
    ax.spines[:].set_color('#0f3460')

    patch = mpatches.Patch(color=color,
                           label=f"{case_display}  |  Distance: {total_dist:.6f}")
    ax.legend(handles=[patch], loc='lower right', facecolor='#0f3460',
              edgecolor='gray', labelcolor='white', fontsize=9.5)

    plt.tight_layout()
    plt.show()

print("✅ Visualization function defined.")


### Algorithm 1 — Route Visualizations (Best / Average / Worst)

In [ ]:
# ─────────────────────────────────────────────
# VISUALIZE: Nearest Neighbor — 3 Cases
# ─────────────────────────────────────────────

for result in greedy_results:
    print(f"\n{'─'*55}")
    print(f"  {result['case'].upper()} CASE — {result['dataset']}")
    print(f"  Cities         : {result['n_cities']}")
    print(f"  Start City     : {result['start']}")
    print(f"  Total Distance : {result['distance']:.6f}")
    print(f"  Execution Time : {result['exec_time']*1000:.3f} ms")
    print(f"{'─'*55}")
    plot_tour(result['coords'], result['tour'], result['dataset'],
              result['distance'], result['case'],
              "Nearest Neighbor (Greedy)", result['exec_time'])


---
## 6. Algorithm 2 — Held-Karp (Dynamic Programming)

### How It Works
The **Held-Karp** algorithm uses bitmask dynamic programming:

1. Represent visited cities as a **bitmask**
2. `dp[mask][i]` = minimum cost to reach city `i` visiting cities in `mask`
3. Build solutions from smaller subsets up to the full set
4. Reconstruct the optimal path via the parent table

### Complexity
| Metric | Value |
|---|---|
| **Time Complexity** | O(2ⁿ · n²) |
| **Space Complexity** | O(2ⁿ · n) |
| **Optimality** | **Guaranteed optimal** |
| **Practical Limit** | ~20 cities max |

### Example Sets
| Case | Dataset | Cities | Reason |
|---|---|---|---|
| **Best** | tiny.csv | 10 | Smallest input — fastest execution |
| **Average** | small.csv subset | 15 | Moderate execution time |
| **Worst** | medium.csv subset | 20 | Largest feasible — slowest execution |

> ⚠️ Running Held-Karp on all cities is computationally infeasible beyond ~20 cities due to exponential growth.


In [ ]:
# ─────────────────────────────────────────────
# ALGORITHM 2: HELD-KARP (DYNAMIC PROGRAMMING)
# ─────────────────────────────────────────────

def held_karp(dist_matrix):
    """
    Held-Karp dynamic programming algorithm for TSP.
    Guarantees the mathematically optimal (shortest) tour.

    Parameters:
        dist_matrix (np.ndarray): Pairwise distance matrix (n x n)

    Returns:
        tour     (list) : Optimal ordered city indices
        min_cost (float): Minimum total tour distance
    """
    n   = len(dist_matrix)
    INF = float('inf')

    # dp[mask][i]     = min cost to reach city i visiting subset 'mask'
    # parent[mask][i] = previous city in the optimal path to i
    dp     = [[INF] * n for _ in range(1 << n)]
    parent = [[-1]  * n for _ in range(1 << n)]

    # Base case: start at city 0 (depot)
    dp[1][0] = 0

    # Fill DP table — build solutions for increasing subset sizes
    for mask in range(1, 1 << n):
        if not (mask & 1):        # Depot must always be in subset
            continue
        for last in range(n):
            if not (mask & (1 << last)):
                continue
            if dp[mask][last] == INF:
                continue
            for nxt in range(n):
                if mask & (1 << nxt):     # Skip already visited
                    continue
                new_mask = mask | (1 << nxt)
                new_cost = dp[mask][last] + dist_matrix[last][nxt]
                if new_cost < dp[new_mask][nxt]:
                    dp[new_mask][nxt] = new_cost
                    parent[new_mask][nxt] = last

    # Find optimal last city before returning to depot
    full_mask = (1 << n) - 1
    min_cost  = INF
    last_city = -1
    for i in range(1, n):
        cost = dp[full_mask][i] + dist_matrix[i][0]
        if cost < min_cost:
            min_cost  = cost
            last_city = i

    # Reconstruct optimal tour
    tour = []
    mask = full_mask
    cur  = last_city
    while cur != -1:
        tour.append(cur)
        prev = parent[mask][cur]
        mask ^= (1 << cur)
        cur   = prev
    tour.reverse()
    tour.append(tour[0])

    return tour, min_cost

print("✅ Held-Karp algorithm defined.")


### Algorithm 2 — Run Best / Average / Worst Cases

In [ ]:
# ─────────────────────────────────────────────
# RUN HELD-KARP — 3 Example Sets
# ─────────────────────────────────────────────

hk_configs = [
    (tiny_cities,   10, "best",    "tiny.csv   — 10 Cities (Full)"),
    (small_cities,  15, "average", "small.csv  — 15 Cities (Subset)"),
    (medium_cities, 20, "worst",   "medium.csv — 20 Cities (Subset)"),
]

hk_results = []

print("=" * 70)
print("  HELD-KARP (DYNAMIC PROGRAMMING) — Results Summary")
print("=" * 70)
print(f"  {'Case':<10} | {'Dataset':<30} | {'Cities':>6} | {'Distance':>12} | {'Time':>10}")
print(f"  {'─'*10}─{'─'*30}─{'─'*8}─{'─'*14}─{'─'*10}")

for base_coords, subset_size, case_label, dataset_name in hk_configs:
    subset_coords = base_coords[:subset_size]
    sub_dist      = compute_distance_matrix(subset_coords)

    t0          = time.time()
    tour, dist  = held_karp(sub_dist)
    exec_time   = time.time() - t0

    hk_results.append({
        "coords"   : subset_coords,
        "tour"     : tour,
        "distance" : dist,
        "case"     : case_label,
        "dataset"  : dataset_name,
        "exec_time": exec_time,
        "n_cities" : subset_size
    })

    print(f"  {case_label.capitalize():<10} | {dataset_name:<30} | {subset_size:>6} | {dist:>12.6f} | {exec_time*1000:>8.3f} ms")

print("=" * 70)


### Algorithm 2 — Route Visualizations (Best / Average / Worst)

In [ ]:
# ─────────────────────────────────────────────
# VISUALIZE: Held-Karp — 3 Cases
# ─────────────────────────────────────────────

for result in hk_results:
    print(f"\n{'─'*55}")
    print(f"  {result['case'].upper()} CASE — {result['dataset']}")
    print(f"  Cities         : {result['n_cities']}")
    print(f"  Total Distance : {result['distance']:.6f}")
    print(f"  Execution Time : {result['exec_time']*1000:.3f} ms")
    print(f"{'─'*55}")
    plot_tour(result['coords'], result['tour'], result['dataset'],
              result['distance'], result['case'],
              "Held-Karp (Dynamic Programming)", result['exec_time'])


---
## 7. Comparison & Analysis


In [ ]:
# ─────────────────────────────────────────────
# FULL COMPARISON: Greedy vs Held-Karp
# ─────────────────────────────────────────────

print("=" * 80)
print("  ALGORITHM COMPARISON TABLE")
print("=" * 80)
print(f"  {'Algorithm':<30} | {'Case':<10} | {'Cities':>6} | {'Distance':>12} | {'Time (ms)':>10}")
print(f"  {'─'*30}─{'─'*10}─{'─'*8}─{'─'*14}─{'─'*10}")

for r in greedy_results:
    print(f"  {'Nearest Neighbor (Greedy)':<30} | {r['case'].capitalize():<10} | {r['n_cities']:>6} | {r['distance']:>12.6f} | {r['exec_time']*1000:>10.3f}")
print(f"  {'─'*30}─{'─'*10}─{'─'*8}─{'─'*14}─{'─'*10}")
for r in hk_results:
    print(f"  {'Held-Karp (DP)':<30} | {r['case'].capitalize():<10} | {r['n_cities']:>6} | {r['distance']:>12.6f} | {r['exec_time']*1000:>10.3f}")
print("=" * 80)

# ── Comparison Bar Charts ──
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor('#1a1a2e')
for ax in axes:
    ax.set_facecolor('#16213e')

cases  = ["Best", "Average", "Worst"]
x      = np.arange(3)
width  = 0.35

g_dists = [r['distance']        for r in greedy_results]
h_dists = [r['distance']        for r in hk_results]
g_times = [r['exec_time'] * 1000 for r in greedy_results]
h_times = [r['exec_time'] * 1000 for r in hk_results]

# Distance
b1 = axes[0].bar(x - width/2, g_dists, width, label='Nearest Neighbor', color='#3498db', alpha=0.85)
b2 = axes[0].bar(x + width/2, h_dists, width, label='Held-Karp',        color='#e74c3c', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(cases, color='white')
axes[0].set_title("Total Distance Comparison", color='white', fontweight='bold', pad=12)
axes[0].set_ylabel("Distance", color='gray')
axes[0].tick_params(colors='gray')
axes[0].spines[:].set_color('#0f3460')
axes[0].legend(facecolor='#0f3460', edgecolor='gray', labelcolor='white')

# Time (log scale)
axes[1].bar(x - width/2, g_times, width, label='Nearest Neighbor', color='#3498db', alpha=0.85)
axes[1].bar(x + width/2, h_times, width, label='Held-Karp',        color='#e74c3c', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(cases, color='white')
axes[1].set_title("Execution Time Comparison (log scale)", color='white', fontweight='bold', pad=12)
axes[1].set_ylabel("Time (ms) — log scale", color='gray')
axes[1].set_yscale('log')
axes[1].tick_params(colors='gray')
axes[1].spines[:].set_color('#0f3460')
axes[1].legend(facecolor='#0f3460', edgecolor='gray', labelcolor='white')

plt.suptitle("Nearest Neighbor vs Held-Karp — Performance Comparison",
             color='white', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


---
## 8. Conclusion

### Summary of Findings

| Criterion | Nearest Neighbor (Greedy) | Held-Karp (DP) |
|---|---|---|
| **Optimality** | ❌ Approximate | ✅ Guaranteed Optimal |
| **Speed** | ✅ Very Fast — O(n²) | ❌ Exponential — O(2ⁿ · n²) |
| **Scalability** | ✅ 1000+ cities easily | ❌ Limited to ~20 cities |
| **Memory** | ✅ O(n) | ❌ O(2ⁿ · n) |
| **Best Use Case** | Real-time large-scale routing | Small critical networks |

### Key Insights

1. **Speed vs. Precision:** Nearest Neighbor runs in milliseconds even for 1000 cities, while Held-Karp's time grows exponentially with each additional city.

2. **Scalability:** In real-world Green Logistics with hundreds of delivery stops, Held-Karp is computationally infeasible. Greedy approaches are essential for real-time routing.

3. **Solution Quality:** Despite being approximate, Nearest Neighbor produces reasonable routes — an acceptable trade-off for large-scale logistics operations.

4. **Practical Recommendation:** For sustainable supply chain management, a **Greedy heuristic** is recommended for large networks. Held-Karp is best reserved for small critical subproblems requiring exact optimal solutions.

---
> *CSC 301: Algorithm Analysis and Design — Academic Year 2025/2026, Imam Abdulrahman Bin Faisal University.*
